# Folketing Open Data – Data Extraction and Exploration

I've made a notebook that explores data from Folketingets Åbne Data (ODA) and constructs a dataset of parliamentary roll-call votes for use in the project.

It follows the data-selection procedure published by Michele (https://www.michelecoscia.com/?page_id=2497) as closely as possible. Their code selects parliamentary cases (`Sag`) with `typeid == 3`, connects these to their procedural steps (`Sagstrin`), and then selects the relevant roll-call votes (`Afstemning`). The new addition is (`titel`) and (`titelkort`) of each case, as these will be used for NLP-based matching with the candidate-test questions. 

The main steps are:

1. Retrieve relevant parliamentary cases (`Sag`) and retain their textual descriptions.
2. Retrieve and connect procedural steps (`Sagstrin`) to the selected cases.
3. Retrieve roll calls (`Afstemning`) and apply the same selection criteria as the original analysis.

I believe the next steps are:

4. Retrieve individual votes (`Stemme`) for the selected roll calls.
5. Connect individual votes to politicians (`Aktør`).
6. Construct a final dataset linking politicians and their votes to the corresponding parliamentary case and its textual description.

Election-period filtering corresponding to FV11, FV15, FV19, and FV22 is a further step to be taken. 

In [36]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://oda.ft.dk/api"

In [37]:
def fetch_all(entity, params=None, page_size=100):
    params = params.copy() if params else {}

    rows = []
    skip = 0

    while True:
        query_params = {
            **params,
            "$top": page_size,
            "$skip": skip
        }

        response = requests.get(
            f"{BASE_URL}/{entity}",
            params=query_params
        )
        response.raise_for_status()

        batch = response.json()["value"]

        if not batch:
            break

        rows.extend(batch)

        if len(batch) < page_size:
            break

        skip += page_size

    return pd.DataFrame(rows)

1. Retrieve relevant parliamentary cases (`Sag`) and retain their textual descriptions.

In [38]:
df_sag = fetch_all(
    "Sag",
    params={
        "$filter": "typeid eq 3",
        "$select": "id,typeid,nummer,titel,titelkort"
    }
)

df_sag.head()

,id,typeid,nummer,titel,titelkort
0,66,3,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
1,68,3,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...
2,69,3,L 107,Forslag til lov om Danmarks Innovationsfond.,Om Danmarks Innovationsfond.
3,70,3,L 109,Forslag til lov om ændring af lov om forskning...,Om konsekvensændringer som følge af lov om Dan...
4,71,3,L 108,Forslag til lov om ændring af lov om teknologi...,Om konsekvensændringer som følge af lov om Dan...


In [39]:
df_sag = df_sag.rename(columns={
    "id": "sagid",
    "typeid": "sagstypeid",
    "nummer": "sag_nummer",
    "titel": "sag_titel",
    "titelkort": "sag_titelkort"
})

In [40]:
print(df_sag.shape)
print(df_sag.columns)
df_sag.head()

(5263, 5)
Index(['sagid', 'sagstypeid', 'sag_nummer', 'sag_titel', 'sag_titelkort'], dtype='object')


,sagid,sagstypeid,sag_nummer,sag_titel,sag_titelkort
0,66,3,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
1,68,3,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...
2,69,3,L 107,Forslag til lov om Danmarks Innovationsfond.,Om Danmarks Innovationsfond.
3,70,3,L 109,Forslag til lov om ændring af lov om forskning...,Om konsekvensændringer som følge af lov om Dan...
4,71,3,L 108,Forslag til lov om ændring af lov om teknologi...,Om konsekvensændringer som følge af lov om Dan...


2. Retrieve and connect procedural steps (`Sagstrin`) to the selected cases.

In [41]:
df_sagstrin = fetch_all(
    "Sagstrin",
    params={
        "$select": "id,dato,typeid,sagid"
    }
)

In [42]:
df_sagstrin = df_sagstrin.rename(columns={
    "id": "sagstrinid",
    "typeid": "sagstrintypeid"
})

In [43]:
df_sagstrin["dato"] = pd.to_datetime(
    df_sagstrin["dato"],
    errors="coerce"
)

In [44]:
df_procedures = df_sagstrin.merge(
    df_sag,
    on="sagid",
    how="inner"
)

In [45]:
df_procedures[
    [
        "sagstrinid",
        "dato",
        "sagstrintypeid",
        "sagid",
        "sag_nummer",
        "sag_titel",
        "sag_titelkort"
    ]
].head(20)

,sagstrinid,dato,sagstrintypeid,sagid,sag_nummer,sag_titel,sag_titelkort
0,135,2014-01-30 10:00:00,31,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
1,136,2014-01-30 00:00:00,32,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
2,137,2014-02-07 10:00:00,12,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
3,142,2014-02-27 00:00:00,14,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
4,144,2014-03-11 13:00:00,15,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
5,146,2014-03-13 10:00:00,17,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
6,147,2014-03-13 00:00:00,37,66,L 105,Forslag til lov om tillægsbevilling for finans...,Om tillægsbevillingsloven for finansåret 2013.
7,150,2014-01-15 13:00:00,31,68,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...
8,151,2014-01-15 00:00:00,32,68,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...
9,152,2014-01-23 10:00:00,12,68,L 110,Forslag til lov om ændring af lov om Det Centr...,Om tildeling af nyt personnummer i særlige til...


3. Retrieve roll calls (`Afstemning`) and apply the same selection criteria as the original analysis.

In [46]:
df_afstemning = fetch_all(
    "Afstemning",
    params={
        "$filter": "typeid eq 1",
        "$select": "id,sagstrinid,kommentar,typeid"
    }
)

In [47]:
df_afstemning = df_afstemning.rename(columns={
    "id": "afstemningid",
    "typeid": "afstemningstypeid"
})

In [48]:
df_afstemning = df_afstemning[
    df_afstemning["kommentar"].isna()
]

In [49]:
df_roll_calls = df_afstemning.merge(
    df_procedures,
    on="sagstrinid",
    how="inner"
)

In [50]:
df_roll_calls[
    [
        "afstemningid",
        "sagstrinid",
        "sagstrintypeid",
        "dato",
        "sagid",
        "sag_nummer",
        "sag_titel",
        "sag_titelkort"
    ]
].head(20)

,afstemningid,sagstrinid,sagstrintypeid,dato,sagid,sag_nummer,sag_titel,sag_titelkort
0,2,4849,17,2014-09-09 09:15:00,1449,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
1,1783,37246,17,2014-10-31 10:00:00,12715,L 4,Forslag til lov om ændring af lov om afgift af...,Om tilbagerulning af forsyningssikkerhedsafgif...
2,2267,38484,17,2014-12-02 13:00:00,13015,L 33,Forslag til lov om ændring af lov om produktio...,Om kompetencebevis.
3,2268,38341,17,2014-12-02 13:00:00,13004,L 19,Forslag til lov om dansk turisme.,Om dansk turisme.
4,2282,38172,17,2014-12-04 10:00:00,12991,L 13,Forslag til lov om ændring af lov om aktiv soc...,Om ændring af formue- og fradragsregler ved ef...
5,2283,38211,17,2014-12-04 10:00:00,12994,L 14,Forslag til lov om ændring af lov om ferie. (F...,Om færre betingelser for optjening af sygeferi...
6,2284,38185,17,2014-12-04 10:00:00,12992,L 15,Forslag til lov om ændring af lov om vikarers ...,Om overførsel af kompetencer til Arbejdsretten.
7,2285,38380,17,2014-12-04 10:00:00,13007,L 17,Forslag til lov om ophævelse af lov om nærings...,Om afskaffelse af næringsbrevsordningen.
8,2286,38353,17,2014-12-04 10:00:00,13005,L 18,Forslag til lov om ændring af konkurrenceloven...,Om ændring af konkurrenceloven m.v.
9,2287,54290,17,2014-12-04 10:00:00,20246,L 46,Forslag til lov om Danmarks Grønne Investering...,Om Danmarks Grønne Investeringsfond.


In [51]:
print("Antal roll calls:", len(df_roll_calls))
print("Unikke afstemninger:", df_roll_calls["afstemningid"].nunique())
print("Unikke sager:", df_roll_calls["sagid"].nunique())

print("\nSagstrin:")
print(df_roll_calls["sagstrintypeid"].value_counts(dropna=False))

print("\nDubletter pr. sag:")
print(
    df_roll_calls["sagid"]
    .value_counts()
    .head(10)
)

Antal roll calls: 2128
Unikke afstemninger: 2128
Unikke sager: 2128

Sagstrin:
sagstrintypeid
17    2128
Name: count, dtype: int64

Dubletter pr. sag:
sagid
103327    1
103791    1
103111    1
104028    1
103494    1
103483    1
103409    1
103495    1
103480    1
103481    1
Name: count, dtype: int64


## From parliamentary cases to final roll-call votes

The extraction connects three levels of the ODA data: parliamentary cases (`Sag`), procedural steps (`Sagstrin`), and roll-call votes (`Afstemning`).

### Parliamentary cases (`Sag`)

Following the selection used in the original analysis, only cases with `Sag.typeid == 3` are retained. In the retrieved data, these correspond to legislative proposals with case numbers beginning with `L`.

This results in 5,261 legislative cases. In addition to the identifiers used in the original analysis, the title (`titel`) and short title (`titelkort`) are retained. These fields provide textual descriptions of the subject of each proposal and will later be used for NLP-based comparison with candidate-test questions.

### Procedural steps (`Sagstrin`)

Each legislative case can contain multiple procedural steps. Joining `Sagstrin` to the selected cases therefore produces several observations for the same `sagid`.

For example, `L 105` appears multiple times with different `sagstrintypeid` values and dates. These observations represent different stages in the parliamentary treatment of the same proposal. One of these stages has `sagstrintypeid == 17`, corresponding to the third reading (`3. behandling`).

At this stage, the dataset therefore represents the parliamentary history of the selected legislative cases rather than one observation per case.

### Roll-call votes (`Afstemning`)

The procedural steps are subsequently joined to `Afstemning`. Following the original analysis, only `Afstemning.typeid == 1` is retained, while observations containing a value in `kommentar` are excluded.

After applying these criteria, the resulting dataset contains 2,131 roll calls.

All 2,131 selected roll calls are associated with `sagstrintypeid == 17`. The filtering procedure therefore results in roll calls taking place at the third reading of the legislative proposals.

Furthermore, there are:

- 2,131 roll calls (`afstemningid`)
- 2,131 unique parliamentary cases (`sagid`)
- no duplicated cases among the selected roll calls

Within this extracted dataset, there is therefore a one-to-one relationship between a selected legislative case and its final roll-call vote:

`Sag (legislative proposal) → Sagstrin (third reading) → Afstemning (roll-call vote)`

This is important for the subsequent analysis because each roll call can be associated unambiguously with the title and short title of a single legislative case. The next step is to connect these roll calls to the individual votes (`Stemme`) cast by parliamentary actors.

## Prepare candidate and parliamentary data

The next stage connects the candidate-test data to the parliamentary voting data.

The candidate-test files contain politicians' names, parties, questions, and answers. The parliamentary roll-call dataset contains the selected final roll calls and their associated legislative cases.

The goal of this stage is to identify the politicians who appear in both datasets and create a mapping between candidate-test names and Folketingets `aktørid`. This identifier can then be used to connect politicians to their individual parliamentary votes.

In [56]:
from pathlib import Path

DATA_DIR = Path("../../temp_data")

candidate_file = DATA_DIR / "candidate_test" / "Kandidattestdata.xlsx"
roll_calls_file = DATA_DIR / "parliament" / "roll_calls.csv"

print(DATA_DIR.resolve())
print(candidate_file.exists())
print(roll_calls_file.exists())

C:\Users\asket\Desktop\Data Science\Bachelor Thesis\Voting_Disconnect\Danish-politics-project\temp_data
False
False


In [60]:
from pathlib import Path

DATA_DIR = Path("../data/temp_data")

candidate_file = DATA_DIR / "candidate_test" / "Kandidattestdata.xlsx"
roll_calls_file = DATA_DIR / "parliament" / "roll_calls.csv"

print("Candidate:", candidate_file.resolve(), candidate_file.exists())
print("Roll calls:", roll_calls_file.resolve(), roll_calls_file.exists())

Candidate: C:\Users\asket\Desktop\Data Science\Bachelor Thesis\Voting_Disconnect\Danish-politics-project\Danish-politics-project\data\temp_data\candidate_test\Kandidattestdata.xlsx True
Roll calls: C:\Users\asket\Desktop\Data Science\Bachelor Thesis\Voting_Disconnect\Danish-politics-project\Danish-politics-project\data\temp_data\parliament\roll_calls.csv True


In [61]:
print("Candidate test shape:", df_candidates.shape)
print(df_candidates.columns.tolist())

print("\nRoll calls shape:", df_roll_calls.shape)
print(df_roll_calls.columns.tolist())

Candidate test shape: (31726, 13)
['id', 'Candidates.Firstname', 'Candidates.Lastname', 'Candidates.Party', 'Candidates.Area', 'gender', 'birthdate', 'Question', 'Answer', 'Answer (text)', 'IsImportant', 'Comment', 'Candidates.Fullname']

Roll calls shape: (2128, 11)
['afstemningid', 'sagstrinid', 'kommentar', 'afstemningstypeid', 'dato', 'sagstrintypeid', 'sagid', 'sagstypeid', 'sag_nummer', 'sag_titel', 'sag_titelkort']


In [62]:
df_candidates["Candidates.Fullname"] = (
    df_candidates["Candidates.Firstname"].str.strip()
    + " "
    + df_candidates["Candidates.Lastname"].str.strip()
)

In [63]:
df_candidate_politicians = (
    df_candidates[
        [
            "Candidates.Fullname",
            "Candidates.Party"
        ]
    ]
    .dropna(subset=["Candidates.Fullname"])
    .drop_duplicates()
    .reset_index(drop=True)
)

In [64]:
print("Unique candidate politicians:", len(df_candidate_politicians))

df_candidate_politicians.head(20)

Unique candidate politicians: 906


,Candidates.Fullname,Candidates.Party
0,Hans Fonsbøl,Alternativet
1,Carsten Sohl,Alternativet
2,Benjamin Strand Andersen,Alternativet
3,Ditte Madvig Evald,Alternativet
4,Torsten Gejl,Alternativet
5,Nikoline Erbs Hillers-Bendtsen,Alternativet
6,Mira Issa Bloch,Alternativet
7,Vinni Kjærgaard Jørgensen,Alternativet
8,Nicklas Gjedsig,Alternativet
9,Jef Seistrup,Alternativet


### Identify political actors

The `Aktør` entity contains several different kinds of actors, including ministries, committees, ministerial roles, institutions, and individual people.

Inspection of the data shows that actors with `typeid == 5` correspond to named individuals. The dataset is therefore restricted to these actors before attempting to match them with candidates from the candidate-test data.

In [65]:
df_aktor = fetch_all(
    "Aktør",
    params={
        "$select": "id,navn,fornavn,efternavn,typeid"
    }
)

In [66]:
print(df_aktor.shape)
print(df_aktor.columns.tolist())

df_aktor.head(20)

(18349, 5)
['id', 'navn', 'fornavn', 'efternavn', 'typeid']


,id,navn,fornavn,efternavn,typeid
0,1,Finansudvalget,None,None,3
1,2,Finansministeriet,None,None,1
2,3,finansministeren,None,None,2
3,4,økonomi- og indenrigsministeren,None,None,2
4,5,Frank Aaen,Frank,Aaen,5
5,6,Folketinget,None,None,11
6,7,Folketinget,Folketinget,,10
7,8,forsvarsministeren,None,None,2
8,9,Forsvarsudvalget,None,None,3
9,10,Grønlandsudvalget,None,None,3


In [67]:
df_aktor = df_aktor.rename(columns={
    "id": "aktørid",
    "navn": "aktør_navn",
    "fornavn": "aktør_fornavn",
    "efternavn": "aktør_efternavn",
    "typeid": "aktør_typeid"
})

In [68]:
df_politicians = df_aktor[
    df_aktor["aktør_typeid"] == 5
].copy()

In [69]:
df_politicians.head(20)

,aktørid,aktør_navn,aktør_fornavn,aktør_efternavn,aktør_typeid
4,5,Frank Aaen,Frank,Aaen,5
11,12,Nicolai Wammen,Nicolai,Wammen,5
12,13,Sara Olsvig,Sara,Olsvig,5
16,17,Christine Antorini,Christine,Antorini,5
17,18,Alex Ahrendtsen,Alex,Ahrendtsen,5
22,23,Bjarne Corydon,Bjarne,Corydon,5
23,24,Karsten Lauritzen,Karsten,Lauritzen,5
27,28,Henrik Sass Larsen,Henrik Sass,Larsen,5
32,33,Holger K. Nielsen,Holger K.,Nielsen,5
33,34,Søren Espersen,Søren,Espersen,5


In [70]:
df_candidate_politicians = df_candidate_politicians.copy()

df_candidate_politicians["Candidates.Fullname"] = (
    df_candidate_politicians["Candidates.Fullname"]
    .str.strip()
)

df_politicians["aktør_navn"] = (
    df_politicians["aktør_navn"]
    .str.strip()
)

In [71]:
df_candidate_actor_match = df_candidate_politicians.merge(
    df_politicians[
        [
            "aktørid",
            "aktør_navn"
        ]
    ],
    left_on="Candidates.Fullname",
    right_on="aktør_navn",
    how="left"
)

In [72]:
candidate_names = set(
    df_candidate_politicians["Candidates.Fullname"]
)

actor_names = set(
    df_politicians["aktør_navn"]
)

candidate_names_in_aktor = candidate_names & actor_names
candidate_names_not_in_aktor = candidate_names - actor_names

print(f"Candidates in Aktør: {len(candidate_names_in_aktor)}")
print(f"Candidates not in Aktør: {len(candidate_names_not_in_aktor)}")

Candidates in Aktør: 289
Candidates not in Aktør: 617


The candidate test includes everyone who participated as a candidate, whereas Aktør is useful to us because it provides the Folketing identifier (aktørid) needed to connect people to actual parliamentary votes. Many of those 617 candidates may simply never have become parliamentary actors.

## Retrieve individual parliamentary votes

The selected roll calls identify the parliamentary decisions included in the analysis, but not the votes cast by individual politicians.

The `Stemme` entity contains one observation per actor and roll call. Each observation links an `afstemningid` to an `aktørid` and records the corresponding vote type.

Only votes associated with the previously selected roll calls are retrieved.

In [73]:
def fetch_stemmer_for_afstemninger(afstemning_ids, batch_size=25):
    rows = []

    for i in range(0, len(afstemning_ids), batch_size):
        batch = afstemning_ids[i:i + batch_size]

        filter_string = " or ".join(
            [f"afstemningid eq {afstemning_id}" for afstemning_id in batch]
        )

        df_batch = fetch_all(
            "Stemme",
            params={
                "$filter": filter_string,
                "$select": "id,typeid,afstemningid,aktørid"
            }
        )

        rows.append(df_batch)

        print(
            f"Processed {min(i + batch_size, len(afstemning_ids))} "
            f"of {len(afstemning_ids)} afstemninger"
        )

    return pd.concat(rows, ignore_index=True)

In [74]:
afstemning_ids = (
    df_roll_calls["afstemningid"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

print("Number of roll calls:", len(afstemning_ids))

Number of roll calls: 2128


In [75]:
df_stemme = fetch_stemmer_for_afstemninger(
    afstemning_ids,
    batch_size=5
)

Processed 5 of 2128 afstemninger
Processed 10 of 2128 afstemninger
Processed 15 of 2128 afstemninger
Processed 20 of 2128 afstemninger
Processed 25 of 2128 afstemninger
Processed 30 of 2128 afstemninger
Processed 35 of 2128 afstemninger
Processed 40 of 2128 afstemninger
Processed 45 of 2128 afstemninger
Processed 50 of 2128 afstemninger
Processed 55 of 2128 afstemninger
Processed 60 of 2128 afstemninger
Processed 65 of 2128 afstemninger
Processed 70 of 2128 afstemninger
Processed 75 of 2128 afstemninger
Processed 80 of 2128 afstemninger
Processed 85 of 2128 afstemninger
Processed 90 of 2128 afstemninger
Processed 95 of 2128 afstemninger
Processed 100 of 2128 afstemninger
Processed 105 of 2128 afstemninger
Processed 110 of 2128 afstemninger
Processed 115 of 2128 afstemninger
Processed 120 of 2128 afstemninger
Processed 125 of 2128 afstemninger
Processed 130 of 2128 afstemninger
Processed 135 of 2128 afstemninger
Processed 140 of 2128 afstemninger
Processed 145 of 2128 afstemninger
Proce

In [76]:
print(df_stemme.shape)
print(df_stemme["afstemningid"].nunique())
print(df_stemme["aktørid"].nunique())
print(df_stemme["typeid"].value_counts(dropna=False))

(363623, 4)
2031
523
typeid
1    190618
3    150822
2     19724
4      2459
Name: count, dtype: int64


In [77]:
missing_roll_calls = sorted(
    set(df_roll_calls["afstemningid"].astype(int))
    - set(df_stemme["afstemningid"].astype(int))
)

print("Missing roll calls:", len(missing_roll_calls))
print(missing_roll_calls[:50])

Missing roll calls: 97
[7901, 7907, 7916, 7917, 7918, 7920, 7921, 7927, 7928, 7929, 7930, 7931, 7944, 7945, 7946, 7947, 7949, 7974, 7975, 7976, 7977, 7978, 7979, 7980, 7981, 7982, 7983, 7984, 7985, 7988, 7989, 7990, 7991, 7998, 7999, 8000, 8001, 8002, 8003, 8004, 8012, 8013, 8014, 8015, 8016, 8017, 8018, 8019, 8020, 8021]


In [ ]:
df_missing_roll_calls = df_roll_calls[
    df_roll_calls["afstemningid"].isin(missing_roll_calls)
]

df_missing_roll_calls[
    [
        "afstemningid",
        "sagid",
        "sag_nummer",
        "sag_titel",
        "dato"
    ]
].head(100)

,afstemningid,sagid,sag_nummer,sag_titel,dato
1249,7901,83404,L 191,Forslag til lov om ændring af sundhedsloven. (...,2020-06-09
1250,7907,81782,L 95,Forslag til lov om ændring af lov om kemikalie...,2020-06-02
1251,7916,83692,L 199,Forslag til lov om ændring af lov om midlertid...,2020-06-09
1252,7917,81783,L 93,Forslag til lov om ejerlejligheder.,2020-06-09
1253,7918,82922,L 184,Forslag til lov om Seniorpensionsenheden.,2020-06-09
...,...,...,...,...,...
2105,10531,103409,L 54,Forslag til lov om ændring af ejendomsskattelo...,2025-12-19
2106,10532,103483,L 63,Forslag til lov om ændring af lov om spil. (In...,2025-12-19
2111,10550,103327,L 50,Forslag til lov om ændring af lov om social se...,2026-02-05
2112,10551,103407,L 55,Forslag til lov om udpegning af retlige repræs...,2026-02-05


In [79]:
missing_id = missing_roll_calls[0]

test_missing = fetch_all(
    "Stemme",
    params={
        "$filter": f"afstemningid eq {missing_id}",
        "$select": "id,typeid,afstemningid,aktørid"
    }
)

print("Afstemning:", missing_id)
print("Rows returned:", len(test_missing))

Afstemning: 7901
Rows returned: 0


In [82]:
vote_type_map = {
    1: "For",
    2: "Imod",
    3: "Fravær",
    4: "Hverken for eller imod"
}

df_stemme["vote"] = df_stemme["typeid"].map(vote_type_map)

In [83]:
test_missing = fetch_all(
    "Stemme",
    params={
        "$filter": "afstemningid eq 1249",
        "$select": "id,typeid,afstemningid,aktørid"
    }
)

print(test_missing.shape)
test_missing.head()

(179, 4)


,id,typeid,afstemningid,aktørid
0,923084,2,1249,221
1,923085,2,1249,158
2,923086,2,1249,351
3,923087,2,1249,17
4,923088,1,1249,49


In [84]:
for afstemning_id in missing_roll_calls[:10]:
    test = fetch_all(
        "Stemme",
        params={
            "$filter": f"afstemningid eq {afstemning_id}",
            "$select": "id,typeid,afstemningid,aktørid"
        }
    )

    print(afstemning_id, len(test))

7901 0
7907 0
7916 0
7917 0
7918 0
7920 0
7921 0
7927 0
7928 0
7929 0


In [85]:
for afstemning_id in [1249, 1250, 1251, 1252, 1253]:
    test = fetch_all(
        "Stemme",
        params={
            "$filter": f"afstemningid eq {afstemning_id}",
            "$select": "id,typeid,afstemningid,aktørid"
        }
    )

    print(afstemning_id, len(test))

1249 179
1250 179
1251 179
1252 0
1253 179


In [86]:
selected_afstemning_ids = set(
    df_roll_calls["afstemningid"]
    .dropna()
    .astype(int)
)

downloaded_afstemning_ids = set(
    df_stemme["afstemningid"]
    .dropna()
    .astype(int)
)

missing_afstemning_ids = sorted(
    selected_afstemning_ids - downloaded_afstemning_ids
)

print("Selected:", len(selected_afstemning_ids))
print("Downloaded:", len(downloaded_afstemning_ids))
print("Missing:", len(missing_afstemning_ids))

print("\nFirst missing IDs:")
print(missing_afstemning_ids[:20])

Selected: 2128
Downloaded: 2031
Missing: 97

First missing IDs:
[7901, 7907, 7916, 7917, 7918, 7920, 7921, 7927, 7928, 7929, 7930, 7931, 7944, 7945, 7946, 7947, 7949, 7974, 7975, 7976]


In [87]:
df_missing = df_roll_calls[
    df_roll_calls["afstemningid"].isin(missing_afstemning_ids)
][
    [
        "afstemningid",
        "sagid",
        "sag_nummer",
        "sag_titel",
        "dato"
    ]
]

df_missing.head(20)

,afstemningid,sagid,sag_nummer,sag_titel,dato
1249,7901,83404,L 191,Forslag til lov om ændring af sundhedsloven. (...,2020-06-09 00:00:00
1250,7907,81782,L 95,Forslag til lov om ændring af lov om kemikalie...,2020-06-02 00:00:00
1251,7916,83692,L 199,Forslag til lov om ændring af lov om midlertid...,2020-06-09 00:00:00
1252,7917,81783,L 93,Forslag til lov om ejerlejligheder.,2020-06-09 00:00:00
1253,7918,82922,L 184,Forslag til lov om Seniorpensionsenheden.,2020-06-09 00:00:00
1254,7920,82923,L 185,Forslag til lov om ændring af lov om godskørse...,2020-06-09 00:00:00
1255,7921,82591,L 167,Forslag til lov om ændring af lov om anerkende...,2020-06-09 00:00:00
1256,7927,83686,L 200,Forslag til lov om ændring af lov om almene bo...,2020-06-16 00:00:00
1257,7928,82041,L 101,Forslag til retsplejelov for Færøerne.,2020-06-16 00:00:00
1258,7929,82039,L 104,Forslag til lov om ændring af retsplejeloven. ...,2020-06-16 00:00:00


In [88]:
for afstemning_id in [1249, 1250, 1251, 1252, 1253]:
    test = fetch_all(
        "Stemme",
        params={
            "$filter": f"afstemningid eq {afstemning_id}",
            "$select": "id,typeid,afstemningid,aktørid"
        }
    )

    print(afstemning_id, len(test))

1249 179
1250 179
1251 179
1252 0
1253 179


In [89]:
def fetch_all(entity, params=None, page_size=100):
    params = params.copy() if params else {}

    # Ensure stable pagination
    if "$orderby" not in params:
        params["$orderby"] = "id asc"

    rows = []
    skip = 0

    while True:
        query_params = {
            **params,
            "$top": page_size,
            "$skip": skip
        }

        response = requests.get(
            f"{BASE_URL}/{entity}",
            params=query_params
        )

        response.raise_for_status()

        batch = response.json()["value"]

        if not batch:
            break

        rows.extend(batch)

        if len(batch) < page_size:
            break

        skip += page_size

    return pd.DataFrame(rows)

In [ ]:
    votes_per_roll_call = (
        df_stemme
        .groupby("afstemningid")
        .size()
        .value_counts()
        .sort_index()
    )

    print(votes_per_roll_call)

179    1965
180      58
181       8
Name: count, dtype: int64


In [91]:
df_vote_counts = (
    df_stemme
    .groupby("afstemningid")
    .size()
    .reset_index(name="n_votes")
)

df_vote_counts["n_votes"].describe()

count    2031.000000
mean      179.036435
std         0.207381
min       179.000000
25%       179.000000
50%       179.000000
75%       179.000000
max       181.000000
Name: n_votes, dtype: float64

In [92]:
df_vote_counts.sort_values("n_votes").head(30)

,afstemningid,n_votes
2030,10588,179
17,2296,179
18,2306,179
19,2307,179
20,2308,179
21,2309,179
22,2310,179
23,2312,179
24,2313,179
25,2314,179


In [93]:
df_stemme = fetch_stemmer_for_afstemninger(
    afstemning_ids,
    batch_size=5
)

Processed 5 of 2128 afstemninger
Processed 10 of 2128 afstemninger
Processed 15 of 2128 afstemninger
Processed 20 of 2128 afstemninger
Processed 25 of 2128 afstemninger
Processed 30 of 2128 afstemninger
Processed 35 of 2128 afstemninger
Processed 40 of 2128 afstemninger
Processed 45 of 2128 afstemninger
Processed 50 of 2128 afstemninger
Processed 55 of 2128 afstemninger
Processed 60 of 2128 afstemninger
Processed 65 of 2128 afstemninger
Processed 70 of 2128 afstemninger
Processed 75 of 2128 afstemninger
Processed 80 of 2128 afstemninger
Processed 85 of 2128 afstemninger
Processed 90 of 2128 afstemninger
Processed 95 of 2128 afstemninger
Processed 100 of 2128 afstemninger
Processed 105 of 2128 afstemninger
Processed 110 of 2128 afstemninger
Processed 115 of 2128 afstemninger
Processed 120 of 2128 afstemninger
Processed 125 of 2128 afstemninger
Processed 130 of 2128 afstemninger
Processed 135 of 2128 afstemninger
Processed 140 of 2128 afstemninger
Processed 145 of 2128 afstemninger
Proce

In [94]:
print("Rows:", len(df_stemme))
print("Unique roll calls:", df_stemme["afstemningid"].nunique())
print("Unique Stemme IDs:", df_stemme["id"].nunique())
print("Duplicate Stemme IDs:", df_stemme["id"].duplicated().sum())

Rows: 363623
Unique roll calls: 2031
Unique Stemme IDs: 363623
Duplicate Stemme IDs: 0


In [109]:
missing_results = []

for i, afstemning_id in enumerate(
    still_missing_afstemning_ids,
    start=1
):
    test = fetch_all(
        "Stemme",
        params={
            "$filter": f"afstemningid eq {afstemning_id}",
            "$select": "id,typeid,afstemningid,aktørid"
        }
    )

    missing_results.append({
        "afstemningid": afstemning_id,
        "n_stemmer": len(test)
    })

    if i % 10 == 0 or i == len(still_missing_afstemning_ids):
        print(f"Checked {i} of {len(still_missing_afstemning_ids)}")

Checked 10 of 97
Checked 20 of 97
Checked 30 of 97
Checked 40 of 97
Checked 50 of 97
Checked 60 of 97
Checked 70 of 97
Checked 80 of 97
Checked 90 of 97
Checked 97 of 97


In [110]:
df_missing_check = pd.DataFrame(missing_results)

df_missing_check["n_stemmer"].value_counts().sort_index()

n_stemmer
0    97
Name: count, dtype: int64

In [111]:
duplicate_afstemninger = df_roll_calls[
    df_roll_calls["afstemningid"].duplicated(keep=False)
].sort_values("afstemningid")

duplicate_afstemninger[
    [
        "afstemningid",
        "sagstrinid",
        "sagid",
        "sag_nummer",
        "dato"
    ]
]

,afstemningid,sagstrinid,sagid,sag_nummer,dato


In [112]:
df_missing.reset_index(drop=True)

,afstemningid,sagid,sag_nummer,sag_titel,dato
0,7901,83404,L 191,Forslag til lov om ændring af sundhedsloven. (...,2020-06-09
1,7907,81782,L 95,Forslag til lov om ændring af lov om kemikalie...,2020-06-02
2,7916,83692,L 199,Forslag til lov om ændring af lov om midlertid...,2020-06-09
3,7917,81783,L 93,Forslag til lov om ejerlejligheder.,2020-06-09
4,7918,82922,L 184,Forslag til lov om Seniorpensionsenheden.,2020-06-09
...,...,...,...,...,...
92,10531,103409,L 54,Forslag til lov om ændring af ejendomsskattelo...,2025-12-19
93,10532,103483,L 63,Forslag til lov om ændring af lov om spil. (In...,2025-12-19
94,10550,103327,L 50,Forslag til lov om ændring af lov om social se...,2026-02-05
95,10551,103407,L 55,Forslag til lov om udpegning af retlige repræs...,2026-02-05


In [113]:
stemme_file = DATA_DIR / "parliament" / "stemme.csv"

df_stemme.to_csv(
    stemme_file,
    index=False
)

print(f"Saved {len(df_stemme):,} rows to:")
print(stemme_file.resolve())

Saved 363,623 rows to:
C:\Users\asket\Desktop\Data Science\Bachelor Thesis\Voting_Disconnect\Danish-politics-project\Danish-politics-project\data\temp_data\parliament\stemme.csv


In [114]:
df_stemme = pd.read_csv(
    DATA_DIR / "parliament" / "stemme.csv"
)

### Local storage of individual votes

The retrieved `Stemme` data are stored locally as `stemme.csv`. Since the ODA API requires a large number of paginated requests to retrieve the individual voting records, subsequent runs of the analysis load this local copy rather than repeating the API extraction.

In [116]:
df_votes_with_names = df_stemme.merge(
    df_politicians[
        ["aktørid", "aktør_navn"]
    ],
    on="aktørid",
    how="left"
)

In [117]:
df_votes_with_names[
    ["afstemningid", "aktørid", "aktør_navn", "typeid"]
].head(20)

,afstemningid,aktørid,aktør_navn,typeid
0,2,158,Eigil Andersen,1
1,2,126,Kim Andersen (udpeget af V),1
2,2,71,Tom Behnke,2
3,2,50,Liselott Blixt,1
4,2,49,Erling Bonnesen,1
5,2,220,Morten Bødskov,1
6,2,183,Bent Bøgsted,1
7,2,85,Özlem Sara Cekic,1
8,2,303,Anita Christensen,1
9,2,94,Peter Christensen,1


In [118]:
vote_type_map = {
    1: "For",
    2: "Imod",
    3: "Fravær",
    4: "Hverken for eller imod"
}

df_votes_with_names["vote"] = (
    df_votes_with_names["typeid"]
    .map(vote_type_map)
)

In [119]:
df_votes_with_names[
    ["afstemningid", "aktørid", "aktør_navn", "vote"]
].head(20)

,afstemningid,aktørid,aktør_navn,vote
0,2,158,Eigil Andersen,For
1,2,126,Kim Andersen (udpeget af V),For
2,2,71,Tom Behnke,Imod
3,2,50,Liselott Blixt,For
4,2,49,Erling Bonnesen,For
5,2,220,Morten Bødskov,For
6,2,183,Bent Bøgsted,For
7,2,85,Özlem Sara Cekic,For
8,2,303,Anita Christensen,For
9,2,94,Peter Christensen,For


In [120]:
voters = (
    df_votes_with_names[
        ["aktørid", "aktør_navn"]
    ]
    .drop_duplicates()
    .sort_values("aktør_navn")
    .reset_index(drop=True)
)

print("Unique voters:", len(voters))
voters.head(50)

Unique voters: 523


,aktørid,aktør_navn
0,15757,Aaja Chemnitz
1,4434,Abbas Razvi
2,18688,Aki-Matilda Høegh-Dam
3,15758,Aleqa Hammond
4,18,Alex Ahrendtsen
5,18703,Alex Vanopslagh
6,19806,Alexander Grandt
7,20395,Alexander Ryle
8,21386,Allan Feldt
9,21362,Amanda Heitmann


In [121]:
print(
    "Votes without matched actor name:",
    df_votes_with_names["aktør_navn"].isna().sum()
)

Votes without matched actor name: 0


In [122]:
df_voter_candidate_match = voters.merge(
    df_candidate_politicians,
    left_on="aktør_navn",
    right_on="Candidates.Fullname",
    how="left"
)

In [123]:
matched_voters = df_voter_candidate_match[
    "Candidates.Fullname"
].notna().sum()

total_voters = len(df_voter_candidate_match)

print(f"Voters in candidate data: {matched_voters}")
print(f"Total parliamentary voters: {total_voters}")
print(f"Match rate: {matched_voters / total_voters:.1%}")

Voters in candidate data: 274
Total parliamentary voters: 523
Match rate: 52.4%


In [124]:
df_unmatched_voters = df_voter_candidate_match[
    df_voter_candidate_match["Candidates.Fullname"].isna()
][
    ["aktørid", "aktør_navn"]
]

df_unmatched_voters.head(50)

,aktørid,aktør_navn
0,15757,Aaja Chemnitz
2,18688,Aki-Matilda Høegh-Dam
3,15758,Aleqa Hammond
6,19806,Alexander Grandt
8,21386,Allan Feldt
9,21362,Amanda Heitmann
11,21360,Anastasia Milthers
14,21349,Anders Kühnau
15,148,Anders Samuelsen
16,17752,Anders Stjernholm


# Combined candidate table across all four election years

In [126]:
sheets = ["FV11", "FV15", "FV19", "FV22"]

candidate_dfs = []

for sheet in sheets:
    df = pd.read_excel(
        candidate_file,
        sheet_name=sheet
    )

    # Handle differences in column names between sheets
    firstname_col = (
        "Candidates.Firstname"
        if "Candidates.Firstname" in df.columns
        else "Firstname"
    )

    lastname_col = (
        "Candidates.Lastname"
        if "Candidates.Lastname" in df.columns
        else "Lastname"
    )

    df["election"] = sheet

    df["Candidates.Fullname"] = (
        df[firstname_col].astype(str).str.strip()
        + " "
        + df[lastname_col].astype(str).str.strip()
    )

    candidate_dfs.append(df)

df_candidates_all = pd.concat(
    candidate_dfs,
    ignore_index=True
)

In [127]:
df_candidates_all[
    [
        "Candidates.Fullname",
        "Candidates.Party",
        "election"
    ]
].head()

,Candidates.Fullname,Candidates.Party,election
0,Abbas Razvi,Radikale Venstre,FV11
1,Abbas Razvi,Radikale Venstre,FV11
2,Abbas Razvi,Radikale Venstre,FV11
3,Abbas Razvi,Radikale Venstre,FV11
4,Abbas Razvi,Radikale Venstre,FV11


In [128]:
df_candidate_politicians_all = (
    df_candidates_all[
        [
            "Candidates.Fullname",
            "Candidates.Party",
            "election"
        ]
    ]
    .dropna(subset=["Candidates.Fullname"])
    .drop_duplicates()
    .reset_index(drop=True)
)

In [129]:
df_voter_candidate_match = voters.merge(
    df_candidate_politicians_all,
    left_on="aktør_navn",
    right_on="Candidates.Fullname",
    how="left"
)

In [130]:
matched_actor_ids = set(
    df_voter_candidate_match.loc[
        df_voter_candidate_match["Candidates.Fullname"].notna(),
        "aktørid"
    ]
)

all_voter_ids = set(voters["aktørid"])

unmatched_actor_ids = all_voter_ids - matched_actor_ids

print("Parliamentary voters:", len(all_voter_ids))
print("Found in at least one candidate test:", len(matched_actor_ids))
print("Not found in candidate tests:", len(unmatched_actor_ids))
print(
    f"Match rate: "
    f"{len(matched_actor_ids) / len(all_voter_ids):.1%}"
)

Parliamentary voters: 523
Found in at least one candidate test: 440
Not found in candidate tests: 83
Match rate: 84.1%


## Finding unmatched politicians. 

Some politicians didn't participate in candidtate test. Maybe these are the ones?

In [131]:
df_unmatched_voters = (
    voters[
        voters["aktørid"].isin(unmatched_actor_ids)
    ]
    .sort_values("aktør_navn")
    .reset_index(drop=True)
)

df_unmatched_voters

,aktørid,aktør_navn
0,15757,Aaja Chemnitz
1,18688,Aki-Matilda Høegh-Dam
2,15758,Aleqa Hammond
3,21386,Allan Feldt
4,21362,Amanda Heitmann
...,...,...
78,288,Torben Hansen (udpeget af S)
79,21372,Trine Birk Andersen
80,21348,Trine Jepsen
81,9780,Tórbjørn Jacobsen


In [132]:
match_summary = (
    df_voter_candidate_match[
        df_voter_candidate_match["Candidates.Fullname"].notna()
    ]
    .groupby("election")["aktørid"]
    .nunique()
)

print(match_summary)

election
FV11    191
FV15    247
FV19    263
FV22    274
Name: aktørid, dtype: int64


In [133]:
import re
import unicodedata

def normalize_name(name):
    if pd.isna(name):
        return None

    name = str(name)

    # Remove parenthetical additions
    name = re.sub(r"\s*\([^)]*\)", "", name)

    # Normalize Unicode representation
    name = unicodedata.normalize("NFC", name)

    # Normalize whitespace and case
    name = " ".join(name.split()).casefold()

    return name

In [134]:
voters["name_normalized"] = (
    voters["aktør_navn"].apply(normalize_name)
)

df_candidate_politicians_all["name_normalized"] = (
    df_candidate_politicians_all["Candidates.Fullname"]
    .apply(normalize_name)
)

In [135]:
df_voter_candidate_match_normalized = voters.merge(
    df_candidate_politicians_all,
    on="name_normalized",
    how="left"
)

In [136]:
matched_actor_ids_normalized = set(
    df_voter_candidate_match_normalized.loc[
        df_voter_candidate_match_normalized[
            "Candidates.Fullname"
        ].notna(),
        "aktørid"
    ]
)

unmatched_actor_ids_normalized = (
    set(voters["aktørid"])
    - matched_actor_ids_normalized
)

print(
    "Matched before normalization:",
    len(matched_actor_ids)
)

print(
    "Matched after normalization:",
    len(matched_actor_ids_normalized)
)

print(
    "Still unmatched:",
    len(unmatched_actor_ids_normalized)
)

Matched before normalization: 440
Matched after normalization: 450
Still unmatched: 73


In [137]:
df_unmatched_normalized = (
    voters[
        voters["aktørid"].isin(
            unmatched_actor_ids_normalized
        )
    ][
        ["aktørid", "aktør_navn"]
    ]
    .sort_values("aktør_navn")
    .reset_index(drop=True)
)

df_unmatched_normalized


,aktørid,aktør_navn
0,15757,Aaja Chemnitz
1,18688,Aki-Matilda Høegh-Dam
2,15758,Aleqa Hammond
3,21386,Allan Feldt
4,21362,Amanda Heitmann
...,...,...
68,21347,Søren Boel Olesen
69,21372,Trine Birk Andersen
70,21348,Trine Jepsen
71,9780,Tórbjørn Jacobsen


In [138]:
from difflib import SequenceMatcher

candidate_names = (
    df_candidate_politicians_all[
        ["Candidates.Fullname", "name_normalized"]
    ]
    .drop_duplicates("name_normalized")
    .dropna()
)

possible_matches = []

for _, voter in df_unmatched_normalized.iterrows():

    voter_normalized = normalize_name(voter["aktør_navn"])

    best_name = None
    best_score = 0

    for _, candidate in candidate_names.iterrows():

        score = SequenceMatcher(
            None,
            voter_normalized,
            candidate["name_normalized"]
        ).ratio()

        if score > best_score:
            best_score = score
            best_name = candidate["Candidates.Fullname"]

    possible_matches.append({
        "aktørid": voter["aktørid"],
        "aktør_navn": voter["aktør_navn"],
        "closest_candidate": best_name,
        "similarity": best_score
    })

df_possible_matches = (
    pd.DataFrame(possible_matches)
    .sort_values("similarity", ascending=False)
    .reset_index(drop=True)
)

df_possible_matches.head(30)

,aktørid,aktør_navn,closest_candidate,similarity
0,18694,Victoria Velasquez,Victoria Velásquez,0.944444
1,21341,Peter Larsen,Peder Larsen,0.916667
2,9780,Tórbjørn Jacobsen,Thorbjørn Jacobsen,0.914286
3,20394,Steffen W. Frølund,Steffen Frølund,0.909091
4,20423,Lars Aagaard,Lars Damgaard,0.880000
5,21365,Sofie F. Villadsen,Sofie Falck Villadsen,0.871795
6,18545,Henrik Old,Henrik Lund,0.857143
7,21398,Pil Christensen,Villum Christensen,0.848485
8,14000,Nick Nielsen,Henrik Nielsen,0.846154
9,217,Lars Christian Lilleholt,Lars Chr. Lilleholt,0.837209


In [139]:
known_actor_ids = [
    18694,
    9780,
    20394,
    21365,
    217,
    20925,
    20375,
    20390,
    15799,
    93,
    21343,
    20352,
    21371
]

In [140]:
df_possible_matches[
    df_possible_matches["aktørid"].isin(known_actor_ids)
][
    [
        "aktørid",
        "aktør_navn",
        "closest_candidate",
        "similarity"
    ]
].sort_values("aktør_navn")

,aktørid,aktør_navn,closest_candidate,similarity
19,20375,Charlotte Munch Pasler,Charlotte Munch,0.810811
18,20925,Dina Raabjerg,Dina Myrup Raabjerg,0.812500
28,20352,Helene Brydensholt,Helene Liliendahl Brydensholt,0.765957
26,21343,Katrine Evelyn Jensen,Kathrine Jensen,0.777778
9,217,Lars Christian Lilleholt,Lars Chr. Lilleholt,0.837209
23,93,Louise Elholm,Louise Schack Elholm,0.787879
21,20390,Mike Villa Fonseca,Mike Fonseca,0.800000
22,15799,Roger Courage Matthisen,Roger Matthisen,0.789474
29,21371,Sinem Dybvad Demir,Sinem Demir,0.758621
5,21365,Sofie F. Villadsen,Sofie Falck Villadsen,0.871795


In [141]:
manual_name_matches = {
    20375: "Charlotte Munch",
    20925: "Dina Myrup Raabjerg",
    20352: "Helene Liliendahl Brydensholt",
    21343: "Kathrine Jensen",
    217:   "Lars Chr. Lilleholt",
    93:    "Louise Schack Elholm",
    20390: "Mike Fonseca",
    15799: "Roger Matthisen",
    21371: "Sinem Demir",
    21365: "Sofie Falck Villadsen",
    20394: "Steffen Frølund",
    9780:  "Thorbjørn Jacobsen",
    18694: "Victoria Velásquez",
}

In [142]:
df_actor_candidate_map = (
    df_voter_candidate_match_normalized[
        df_voter_candidate_match_normalized[
            "Candidates.Fullname"
        ].notna()
    ][
        [
            "aktørid",
            "aktør_navn",
            "Candidates.Fullname",
            "Candidates.Party",
            "election"
        ]
    ]
    .copy()
)

df_actor_candidate_map["match_method"] = "normalized_exact"

In [143]:
manual_matches = []

for aktørid, candidate_name in manual_name_matches.items():

    candidate_rows = df_candidate_politicians_all[
        df_candidate_politicians_all["Candidates.Fullname"]
        == candidate_name
    ]

    actor_name = voters.loc[
        voters["aktørid"] == aktørid,
        "aktør_navn"
    ].iloc[0]

    for _, candidate in candidate_rows.iterrows():
        manual_matches.append({
            "aktørid": aktørid,
            "aktør_navn": actor_name,
            "Candidates.Fullname": candidate["Candidates.Fullname"],
            "Candidates.Party": candidate["Candidates.Party"],
            "election": candidate["election"],
            "match_method": "manual"
        })

df_manual_matches = pd.DataFrame(manual_matches)

In [144]:
df_actor_candidate_map = pd.concat(
    [
        df_actor_candidate_map,
        df_manual_matches
    ],
    ignore_index=True
).drop_duplicates()

In [145]:
print(
    "Matched parliamentary actors:",
    df_actor_candidate_map["aktørid"].nunique()
)

print(
    "Total parliamentary actors:",
    voters["aktørid"].nunique()
)

print(
    "Still unmatched:",
    voters["aktørid"].nunique()
    - df_actor_candidate_map["aktørid"].nunique()
)

Matched parliamentary actors: 463
Total parliamentary actors: 523
Still unmatched: 60


In [146]:
vote_type_map = {
    1: "For",
    2: "Imod",
    3: "Fravær",
    4: "Hverken for eller imod"
}

df_stemme["vote"] = df_stemme["typeid"].map(vote_type_map)

In [147]:
matched_actor_ids = set(
    df_actor_candidate_map["aktørid"]
)

df_votes_matched = df_stemme[
    df_stemme["aktørid"].isin(matched_actor_ids)
].copy()

In [148]:
df_votes_matched = df_votes_matched.merge(
    df_roll_calls[
        [
            "afstemningid",
            "dato",
            "sagid",
            "sag_nummer",
            "sag_titel",
            "sag_titelkort"
        ]
    ],
    on="afstemningid",
    how="inner",
    validate="many_to_one"
)

In [149]:
print("Rows:", len(df_votes_matched))
print(
    "Politicians:",
    df_votes_matched["aktørid"].nunique()
)
print(
    "Roll calls:",
    df_votes_matched["afstemningid"].nunique()
)
print(
    "Cases:",
    df_votes_matched["sagid"].nunique()
)

df_votes_matched[
    [
        "aktørid",
        "vote",
        "dato",
        "sag_nummer",
        "sag_titel",
        "sag_titelkort"
    ]
].head(20)

Rows: 354092
Politicians: 463
Roll calls: 2031
Cases: 2031


,aktørid,vote,dato,sag_nummer,sag_titel,sag_titelkort
0,158,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
1,126,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
2,71,Imod,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
3,50,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
4,49,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
5,220,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
6,183,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
7,85,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
8,303,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...
9,94,For,2014-09-09 09:15:00,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...


In [150]:
actor_candidate_file = (
    DATA_DIR / "parliament" / "actor_candidate_map.csv"
)

df_actor_candidate_map.to_csv(
    actor_candidate_file,
    index=False
)

print(f"Saved {len(df_actor_candidate_map):,} rows to:")
print(actor_candidate_file.resolve())

Saved 1,021 rows to:
C:\Users\asket\Desktop\Data Science\Bachelor Thesis\Voting_Disconnect\Danish-politics-project\Danish-politics-project\data\temp_data\parliament\actor_candidate_map.csv


In [151]:
df_actor_candidate_map = pd.read_csv(
    DATA_DIR / "parliament" / "actor_candidate_map.csv"
)

In [152]:
politicians_file = (
    DATA_DIR / "parliament" / "politicians.csv"
)

df_politicians.to_csv(
    politicians_file,
    index=False
)